# Notebook 2: Feature Store

This notebook creates a Dynamic Table that serves as an online feature store, auto-refreshing every hour. It joins sensor readings with production data to compute rolling health features per well.

**Pipeline Position:** Transforms raw sensor data (from Notebook 1) into ML-ready features consumed by the training job (Notebook 3) and inference service (Notebook 4).

**Features computed:**
- Latest sensor readings + pressure differential (pump efficiency proxy)
- 7-day rolling averages and standard deviations
- Production metrics: water cut %, gas-oil ratio (GOR)
- Z-scores for anomaly detection (amps, temperature, vibration deviation)

**Output:** `ENERGY_DEMO.WELLS.WELL_HEALTH_FEATURES` — 40 rows (latest feature snapshot per well)

**Approach:** Uses Snowpark DataFrames to build feature logic, then registers the result as a Dynamic Table via `create_or_replace_dynamic_table()`.

## 1. Connect to Snowflake

Establish a Snowpark session using the active notebook connection.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark import Window

session = get_active_session()

DATABASE = "ENERGY_DEMO"
SCHEMA = "WELLS"

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(f"Connected to {DATABASE}.{SCHEMA}")

## 2. Build Feature DataFrame with Snowpark

Construct the feature logic using Snowpark DataFrames:

1. **latest_sensors** — Most recent reading per well (using `row_number` + filter)
2. **sensor_stats** — 7-day rolling averages and standard deviations for anomaly baselines
3. **prod_stats** — Production averages, water cut percentage, and gas-oil ratio

The final DataFrame combines these into a single wide row per well, including z-scores that flag how far current readings deviate from recent history.

In [ ]:
well_sensors = session.table("WELL_SENSORS")
well_production = session.table("WELL_PRODUCTION")

window_latest = Window.partition_by("API_NO").order_by(F.col("READING_TS").desc())
latest_sensors = (
    well_sensors.with_column("RN", F.row_number().over(window_latest))
    .filter(F.col("RN") == 1)
    .drop("RN")
)

sensor_with_max_ts = latest_sensors.select(
    F.col("API_NO").alias("LS_API_NO"), F.col("READING_TS").alias("MAX_TS")
)

sensor_7d = (
    well_sensors.join(
        sensor_with_max_ts, well_sensors["API_NO"] == sensor_with_max_ts["LS_API_NO"]
    )
    .filter(F.col("READING_TS") >= F.dateadd("day", F.lit(-7), F.col("MAX_TS")))
    .drop("LS_API_NO", "MAX_TS")
)

sensor_stats = sensor_7d.group_by("API_NO").agg(
    F.avg("INTAKE_PRESSURE_PSI").alias("AVG_INTAKE_PRESSURE_7D"),
    F.stddev("INTAKE_PRESSURE_PSI").alias("STD_INTAKE_PRESSURE_7D"),
    F.avg("DISCHARGE_PRESSURE_PSI").alias("AVG_DISCHARGE_PRESSURE_7D"),
    F.avg("MOTOR_TEMP_F").alias("AVG_MOTOR_TEMP_7D"),
    F.stddev("MOTOR_TEMP_F").alias("STD_MOTOR_TEMP_7D"),
    F.avg("MOTOR_AMPS").alias("AVG_MOTOR_AMPS_7D"),
    F.stddev("MOTOR_AMPS").alias("STD_MOTOR_AMPS_7D"),
    F.avg("VIBRATION_IPS").alias("AVG_VIBRATION_7D"),
    F.stddev("VIBRATION_IPS").alias("STD_VIBRATION_7D"),
    F.avg("FREQUENCY_HZ").alias("AVG_FREQUENCY_7D"),
)

max_prod_date = well_production.select(
    F.max("PRODUCTION_DATE").alias("MAX_DATE")
).collect()[0]["MAX_DATE"]

prod_stats = (
    well_production.filter(
        F.col("PRODUCTION_DATE") >= F.dateadd("day", F.lit(-7), F.lit(max_prod_date))
    )
    .group_by("API_NO")
    .agg(
        F.avg("OIL_BBL").alias("AVG_OIL_7D"),
        F.avg("GAS_MCF").alias("AVG_GAS_7D"),
        F.avg("WATER_BBL").alias("AVG_WATER_7D"),
        F.avg("RUNTIME_HOURS").alias("AVG_RUNTIME_7D"),
    )
    .with_column(
        "WATER_CUT_PCT",
        F.col("AVG_WATER_7D")
        / F.iff(
            F.col("AVG_OIL_7D") + F.col("AVG_WATER_7D") == 0,
            F.lit(None),
            F.col("AVG_OIL_7D") + F.col("AVG_WATER_7D"),
        ),
    )
    .with_column(
        "GOR",
        F.col("AVG_GAS_7D")
        / F.iff(F.col("AVG_OIL_7D") == 0, F.lit(None), F.col("AVG_OIL_7D")),
    )
)

features_df = (
    latest_sensors.select(
        "API_NO",
        "WELL_NAME",
        F.col("READING_TS").alias("FEATURE_TS"),
        "INTAKE_PRESSURE_PSI",
        "DISCHARGE_PRESSURE_PSI",
        (F.col("DISCHARGE_PRESSURE_PSI") - F.col("INTAKE_PRESSURE_PSI")).alias(
            "PRESSURE_DIFFERENTIAL"
        ),
        "MOTOR_TEMP_F",
        "MOTOR_AMPS",
        "VIBRATION_IPS",
        "WELLHEAD_PRESSURE_PSI",
        "WELLHEAD_TEMP_F",
        "FREQUENCY_HZ",
    )
    .join(sensor_stats, "API_NO", "left")
    .join(prod_stats, "API_NO", "left")
    .with_column(
        "AMP_ZSCORE",
        (F.col("MOTOR_AMPS") - F.col("AVG_MOTOR_AMPS_7D"))
        / F.iff(
            F.col("STD_MOTOR_AMPS_7D") == 0, F.lit(None), F.col("STD_MOTOR_AMPS_7D")
        ),
    )
    .with_column(
        "TEMP_ZSCORE",
        (F.col("MOTOR_TEMP_F") - F.col("AVG_MOTOR_TEMP_7D"))
        / F.iff(
            F.col("STD_MOTOR_TEMP_7D") == 0, F.lit(None), F.col("STD_MOTOR_TEMP_7D")
        ),
    )
    .with_column(
        "VIBRATION_ZSCORE",
        (F.col("VIBRATION_IPS") - F.col("AVG_VIBRATION_7D"))
        / F.iff(F.col("STD_VIBRATION_7D") == 0, F.lit(None), F.col("STD_VIBRATION_7D")),
    )
)

print(f"Feature DataFrame columns: {len(features_df.columns)}")
print(features_df.columns)

## 3. Register as Dynamic Table

Register the Snowpark DataFrame as a Dynamic Table with a 1-hour refresh lag. Then trigger an immediate refresh and verify we get 40 rows (one per well).

In [ ]:
features_df.create_or_replace_dynamic_table(
    name="WELL_HEALTH_FEATURES",
    warehouse="COMPUTE_WH",
    lag="1 hour",
)
print("Dynamic Table WELL_HEALTH_FEATURES created.")

session.sql("ALTER DYNAMIC TABLE WELL_HEALTH_FEATURES REFRESH").collect()
count = session.table("WELL_HEALTH_FEATURES").count()
print(f"Rows: {count} (expected: 40, one per well)")

## 4. Preview Features

Spot-check key features for a few wells: raw sensor values, production ratios, and anomaly z-scores.

In [ ]:
session.table("WELL_HEALTH_FEATURES").select(
    "WELL_NAME",
    "INTAKE_PRESSURE_PSI",
    "PRESSURE_DIFFERENTIAL",
    "MOTOR_TEMP_F",
    "MOTOR_AMPS",
    "VIBRATION_IPS",
    "WATER_CUT_PCT",
    "GOR",
    "AMP_ZSCORE",
    "TEMP_ZSCORE",
    "VIBRATION_ZSCORE",
).sort("WELL_NAME").limit(10).to_pandas()